In [9]:
# 一、准备文档
from langchain_text_splitters import MarkdownHeaderTextSplitter
from dotenv import load_dotenv
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings
from pymilvus import Function,FunctionType

# 加载配置文件
load_dotenv()
print(f"> 配置加载成功")
# 创建向量模型
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key="sk-ws-H.YIEXDIY.vFkq.MEYCIQC_yxHSkXG8uXAS3p_D7J31I9gabZmVixTG96X8BUVlegIhALglLtL4-Xyai0SoeaLMM2uivtrWeFejakoJNqhtIXE8"
)

def prepare_docs():
    # =============1.准备文档================
    with open('../../day08/resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    # 为了方便查看，给每个Document加一个id，从1开始递增
    # 也可以让 milvus 自动生成
    # - LangChain写入Milvus的文档如果自定义ID，则ID必须是字符串类型！
    # - 也可以不指定ID，则ID由Milvus生成，为INT64类型
    #for i, doc in enumerate(docs, start=1):
        #doc.id = str(i)
    return docs

docs = prepare_docs()
print(f"> 创建文档成功")

#===========================LangChain-Milvus连接===========================
#创建Milvus
client = Milvus(
    embedding_function= embeddings,
    collection_name = "langchain_collection",
    auto_id = True,
    builtin_function = BM25BuiltInFunction(
        analyzer_params = {"type":"chinese"}
    ),
    vector_field = ["dense","sparse"],
    connection_args = {"uri": settings.rag.milvus_url},
    drop_old = True,
)
print(f"> 创建Milvus成功")
#添加数据
client.add_documents(docs)
print(f"> 数据已添加成功")
#删除数据
#client.delete(ids = 470244795427663400)
#print(f"> 数据\" {docs} \"已删除成功")

#用户提问:
query = "教育的负向作用是什么"
#重排序检索:
#   使用reranker重排序
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",# ranker名词，唯一即可
        input_field_names=["text"],# 原始文档字段
        function_type=FunctionType.RERANK,# ranker类型，这里是固定值
        params={
            "reranker":"model",             # rerank类型，基于模型rerank,也就是cross-encoder
            "model_name":"gte-rerank-v2",   # rerank模型名称
            "provider":"ali",               # rerank模型提供者
            "queries": queries,             # 查询条件
            "max_client_batch_size":5       # 向模型发送请求时的批处理限制
            },
    )
res = client.similarity_search(
    query=query ,
    k = 5,
    reranker = create_cross_encoder_ranker([query])
)
print("===================== 重 排 检 索 结 果 ======================")
print(res)

> 配置加载成功
> 创建文档成功
> 创建Milvus成功
> 数据已添加成功
===================== 重 排 检 索 结 果 ======================
[Document(metadata={'h1': '第二章 教育基本原理', 'h2': '第一节 教育的功能', 'h3': '（一）个体发展功能和社会发展功能', 'pk': 470244795427663498}, page_content='# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。'), Document(metadata={'h1': '第二章 教育基本原理', 'h2': '第二节 教育和社会的关系', 'h3': '（二）经济与教育的关系', 'pk': 470244795427663500}, page_content='### （二）经济与教育的关系  \n经济发展对教育的影响：经济发展的规模和速度制约着教育发展的规模和速度；经济发展水平制约着教育事业发展的结构和布局；经济发展通过对人才和知识的需求制约着教育目标的设定；经济发展水平制约着学校的教育内容、方式方法手段和教学组织形式；经济发展制约着教育事业发展的基础。  \n教育在经济发展中的作用表现为：教育为经济的发展提供良好基础；教育再生产劳动力；教育是科学技术再生产的手段。'), Document(metadata={'h1': '第二章 教育基本原理', 'h2': '第二节 教育和社会的关系', 'h3': '（六）生产力与教育的关系', 'pk': 470244795427663504}, page_content='### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规